In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('student_data.csv')
df.head()

,StudentID,Day,Mood,ShirtColor
0,1,1,H,R
1,1,2,H,R
2,1,3,S,B
3,1,4,S,B
4,1,5,H,R


In [3]:
day_1 = df[df['Day'] == 1]
day_1_student_count = len(day_1)

In [4]:
initial_counts = day_1['Mood'].value_counts()
initial_counts

Mood
H    3
S    2
Name: count, dtype: int64

In [5]:
p1_probs = (initial_counts / day_1_student_count).to_dict()
p1_probs

{'H': 0.6, 'S': 0.4}

In [6]:
from collections import Counter
moods = df['Mood'].tolist()
transition_m = []
t_counter = Counter()

In [7]:
for i in range(len(moods) - 1):
    new_student_start = (1+i) % 20 == 0
    if not new_student_start:
        m1 = moods[i]
        m2 = moods[i + 1]

        transition_m.append((m1, m2))
        t_counter.update([m1])

In [8]:
pair_counts = Counter(transition_m)

In [9]:
pair_counts

Counter({('H', 'H'): 36, ('S', 'S'): 22, ('H', 'S'): 19, ('S', 'H'): 18})

In [12]:
states = df['Mood'].unique().tolist()
states

['H', 'S']

In [13]:
trans_probs = {s1: {s2 : 0.0 for s2 in states} for s1 in states}
trans_probs

{'H': {'H': 0.0, 'S': 0.0}, 'S': {'H': 0.0, 'S': 0.0}}

In [15]:
for (s1, s2), count in pair_counts.items():
    if t_counter[s1] > 0:
        trans_probs[s1][s2] = count / t_counter[s1]


In [16]:
trans_probs

{'H': {'H': 0.6545454545454545, 'S': 0.34545454545454546},
 'S': {'H': 0.45, 'S': 0.55}}

In [17]:
total_mood = df['Mood'].value_counts()

joints_count = df.groupby(['Mood'])['ShirtColor'].value_counts()

In [21]:
total_mood

Mood
H    57
S    43
Name: count, dtype: int64

In [18]:
joints_count

Mood  ShirtColor
H     R             41
      G             16
S     B             37
      G              6
Name: count, dtype: int64

In [19]:
observations = df['ShirtColor'].unique().tolist()
observations

['R', 'B', 'G']

In [20]:
emmision_skeleton = {s : {o : 0.0 for o in observations} for s in states}
emmision_skeleton

{'H': {'R': 0.0, 'B': 0.0, 'G': 0.0}, 'S': {'R': 0.0, 'B': 0.0, 'G': 0.0}}

In [22]:
for s in states:
    total_count = total_mood[s]
    for o in observations:
        if (s, o) in joints_count:
            emmision_skeleton[s][o] = joints_count[(s,o)] / total_count
emmision_skeleton

{'H': {'R': 0.7192982456140351, 'B': 0.0, 'G': 0.2807017543859649},
 'S': {'R': 0.0, 'B': 0.8604651162790697, 'G': 0.13953488372093023}}

In [23]:
possible_sequences = [
    ('H', 'H', 'H'),
    ('H', 'H', 'S'),
    ('H', 'S', 'H'),
    ('H', 'S', 'S'),
    ('S', 'H', 'H'),
    ('S', 'H', 'S'),
    ('S', 'S', 'H'),
    ('S', 'S', 'S')
]

In [25]:
results = []
max_prob = 0
max_sequence = None
observed_sequences = ['R', 'G', 'B']
o1, o2, o3 = observed_sequences

In [26]:
for sequence in possible_sequences:
    m1, m2, m3 = sequence
    prob = p1_probs[m1] * emmision_skeleton[m1][o1] * trans_probs[m1][m2] * emmision_skeleton[m2][o2] * trans_probs[m2][m3] * emmision_skeleton[m3][o3]
    results.append((sequence, prob))

    if prob > max_prob:
        max_prob = prob
        max_sequence = sequence


In [27]:
print(observed_sequences)
print('\n')
print(max_sequence)
print('\n')
print(max_prob)

['R', 'G', 'B']


('H', 'H', 'S')


0.02357053117128782
